# Lesson 3: Your first neural network with PyTorch 🧠

**Do Lesson 1 and Lesson 2 first.** We explain each new term before using it.

### What is a neural network?
A neural network is a function with adjustable numbers called **weights** and **biases**. During **training**, it changes those numbers to make better predictions.

A **layer** takes some numbers in and produces numbers out. An **activation function** like `ReLU` allows a model to learn bends and more complicated patterns (not just straight lines).

Today we will predict **Sales** from **TV, Radio, Newspaper** and compare with linear regression. On only 200 rows, a neural network may **not** win — learning to compare fairly is the goal.

## 1. Load data, and create a fair train/test split

We will use the *same* random split as Lesson 2. The test set is our final exam and must not be used to train.

**Scaling:** A TV budget number may be much bigger than a Radio budget number. Scaling puts inputs on comparable ranges; we learn the scaling values from the training set only to avoid cheating.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Reproducible random numbers
np.random.seed(42)
torch.manual_seed(42)

path = Path('data/Advertising.csv')
if not path.exists():
    path = Path('../data/Advertising.csv')
df = pd.read_csv(path).drop(columns=['Unnamed: 0'], errors='ignore')
X = df[['TV','Radio','Newspaper']]
y = df['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Training examples:', len(X_train), '| Test examples:',len(X_test))

## 2. What are tensors?

A **tensor** is just a container of numbers used by PyTorch (similar to a NumPy array). We need two-dimensional input tensors: rows are examples, columns are features.

The target must also have two dimensions here (a column of Sales values).

In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy().reshape(-1,1), dtype=torch.float32)
print('Features shape:', X_train_t.shape)
print('Target shape:', y_train_t.shape)
print('One example:',X_train_t[0])

## 3. Build a tiny neural network

- `Linear(3, 8)` = 3 inputs go to 8 small computation units (neurons)
- `ReLU()` = activation function; it introduces nonlinearity
- `Linear(8, 1)` = combine the 8 numbers into one prediction (**Sales**)

This is called a **feed-forward network**. You don't need to memorize the code yet.

**Your turn:** Why does the first layer have 3 inputs instead of 4?

In [ ]:
network = torch.nn.Sequential(
    torch.nn.Linear(3, 8),
    torch.nn.ReLU(),
    torch.nn.Linear(8, 1),
)
print(network)

## 4. How does it learn?

First, the network guesses Sales. We compare the guess to the real Sales using a **loss** (mean squared error, or MSE). Next PyTorch calculates how each weight affected the loss (these derivatives are called **gradients**). Finally the **optimizer** updates weights to try to reduce loss.

One pass through the entire training dataset is an **epoch**.

We will repeat this 1,000 times. The learning rate controls the size of each update.

In [ ]:
loss_fn = torch.nn.MSELoss()  # Average squared prediction error
optimizer = torch.optim.Adam(network.parameters(), lr=0.01)
loss_history = []

for epoch in range(1000):
    network.train()
    predicted = network(X_train_t)        # 1. Forward pass: guess
    loss = loss_fn(predicted, y_train_t)   # 2. Compare with answers
    optimizer.zero_grad()                 # Clear old gradients
    loss.backward()                      # 3. Calculate gradients
    optimizer.step()                     # 4. Update weights
    loss_history.append(loss.item())

    if (epoch + 1) % 200 == 0:
        print(f'Epoch {epoch+1}: training MSE = {loss.item():.3f}')

## 5. What does learning look like?

Loss going down means the model is fitting the **training** data better. It does NOT prove it will do well on unseen data. We check that next.

In [ ]:
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('Training MSE')
plt.title('Learning curve')
plt.show()

## 6. Compare neural network and linear regression fairly

We use the SAME held-out test examples for both. **MAE** (average absolute error) is easier to interpret than MSE: it tells us how far off predictions are on average, in Sales units. Lower is better.

We do not train or tune the neural network on the test set.

In [ ]:
network.eval()
with torch.no_grad():  # Prediction mode: no gradients needed
    nn_predictions = network(X_test_t).numpy().ravel()

baseline = LinearRegression()
baseline.fit(X_train_scaled, y_train)
linear_predictions = baseline.predict(X_test_scaled)

nn_mae = mean_absolute_error(y_test, nn_predictions)
linear_mae = mean_absolute_error(y_test, linear_predictions)
print(f'Linear regression test MAE: {linear_mae:.3f}')
print(f'Neural network test MAE:  {nn_mae:.3f}')
print('Lower MAE is better on this test set.')

## 7. Look at actual predictions

**Your turn:** Which model has lower test MAE? Does training a more complicated model always guarantee better predictions?

In [ ]:
results = pd.DataFrame({'Actual Sales':y_test.to_numpy(), 'Linear prediction':linear_predictions, 'Neural network prediction':nn_predictions})
results.head(10).round(2)

## Challenges ✍️ (one at a time)

1. **Explain in your own words:** What does a weight do? What is loss?
2. Change the number of neurons from `8` to `4` and rerun the model/training/evaluation cells. Did test MAE improve?
3. Try changing training epochs from `1000` to `300`. What happens to training loss?
4. Why is it important to keep test examples separate?
5. **Important:** this is a tiny dataset. Can you think of reasons a simpler model might be safer or work better?

**Extra credit (optional):** Learn what overfitting means. If you change the model many times based on test MAE, you are indirectly using the test set to choose your model; for a more rigorous experiment, create a separate validation set.

🎉 You built and evaluated your first PyTorch neural network!